# STEP 1: 1_preprocess_data.ipynb

Press SHIFT + ENTER to run code

### USER INPUT!
Specify where to store the csvs. change the data_root_dir

In [1]:
## Define project
project_name = ''

## Say where your data holding your DLC-analyzed CSVs is stored
    # i.e. Apple: '/Users/justinjames/LUPE_Corder-Lab/'+project_XXX+'/dlc_csvs'
data_root_dir = ''

## Breakdown how your data is organized in the folders-- name of folders that are groups? within groups, name condition folders
    # i.e. groups = ['Group1', 'Group2','Group3']
    # i.e. conditions = ['control','experiment']
groups = ['','']
conditions = ['','']

### Updating meta.py for project

In [2]:
import os

def update_meta_file(project_name):
    meta_file_path = '../utils/meta.py'
    
    groups_var = f"groups_{project_name} = {groups}"
    conditions_var = f"conditions_{project_name} = {conditions}"
    
    # Read the current contents of the meta file
    if os.path.exists(meta_file_path):
        with open(meta_file_path, 'r') as file:
            lines = file.readlines()
    else:
        lines = []

    # Check if the variables are already defined and update them if necessary
    groups_defined = False
    conditions_defined = False
    for i, line in enumerate(lines):
        if line.startswith(f"groups_{project_name} ="):
            lines[i] = groups_var + '\n'
            groups_defined = True
        elif line.startswith(f"conditions_{project_name} ="):
            lines[i] = conditions_var + '\n'
            conditions_defined = True

    # If the variables are not defined, add them to the end of the file
    if not groups_defined:
        lines.append(groups_var + '\n')
    if not conditions_defined:
        lines.append(conditions_var + '\n')

    # Write the updated contents back to the meta file
    with open(meta_file_path, 'w') as file:
        file.writelines(lines)
    
    print(f'Updated {meta_file_path} with project-specific groups and conditions.')

# Example usage
update_meta_file(project_name)

Updated ../utils/meta.py with project-specific groups and conditions.


### Main Code: store all data in dictionary
WAIT UNTIL PROCESSING DATA FINISHES

In [3]:
###### RUN DEPENDENCIES ######
import glob
import pickle
import os
import sys
if not os.path.join(os.path.abspath(''), '../') in sys.path:
    sys.path.append(os.path.join(os.path.abspath(''), '../'))
import numpy as np
import pandas as pd
from tqdm import notebook
from utils.feature_utils import filter_pose_noise

###### MAIN CODE ######
filenames = {key: [] for key in groups}
data = {key: [] for key in groups}
for group in notebook.tqdm(groups):
    filenames[group] = {key: [] for key in conditions}
    data[group] = {key: [] for key in conditions}
    for condition in notebook.tqdm(conditions):
        
        filenames[group][condition] = glob.glob(str.join('/', 
                                                              (data_root_dir,
                                                               f'{group}', 
                                                               f'{condition}', 
                                                               '*.csv')))
        data[group][condition] = {os.path.splitext(os.path.basename(csv))[0]: [] 
                                  for csv in filenames[group][condition]}
        
        for csv in notebook.tqdm(filenames[group][condition]):
            temp_df = pd.read_csv(csv, header=[0, 1, 2, 3], sep=",", index_col=0)
            selected_pose_idx = np.arange(temp_df.shape[1])
            idx_llh = selected_pose_idx[2::3]
            # the loaded sleap file has them too, so exclude for both
            idx_selected = [i for i in selected_pose_idx if i not in idx_llh]
            currdf_filt, _ = filter_pose_noise(temp_df, idx_selected=idx_selected, idx_llh=idx_llh, 
                                               llh_value=0.1)
            file_name = os.path.splitext(os.path.basename(csv))[0]
            data[group][condition][file_name] = currdf_filt

###### WAIT UNTIL PROCESSING DATA FINISHES ######

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [4]:
###### STORE ALL DATA IN DICTIONARY ######
base_dir = "../processed_dataset"
directory = os.path.join(base_dir, project_name)
os.makedirs(directory, exist_ok=True)

raw_data_pkl_filename = os.path.join(directory, f"raw_data_{project_name}.pkl")
with open(raw_data_pkl_filename, "wb") as f:
    pickle.dump(data, f)

print(f"{raw_data_pkl_filename} is created and saved!")

from datetime import datetime

###### STORE PROJECT INFO AND DATA STRUCTURE IN .TXT FILE ######
def write_project_info_file(project_dir, project_name, groups, conditions, filenames):
    meta_path = os.path.join(project_dir, f"project_info_{project_name}.txt")
    with open(meta_path, "w") as f:
        # Header
        f.write(f"Project: {project_name}\n")
        f.write(f"Created/Updated: {datetime.now().isoformat()}\n\n")
        # Groups
        f.write("Groups:\n")
        for g in groups:
            f.write(f"- {g}\n")
        f.write("\n")
        # Conditions
        f.write("Conditions:\n")
        for c in conditions:
            f.write(f"- {c}\n")
        f.write("\n")
        # Files by Group and Condition
        f.write("Files by Group and Condition:\n\n")
        for g in groups:
            f.write(f"[{g}]\n")
            for c in conditions:
                f.write(f"  {c}:\n")
                for csv_path in filenames[g][c]:
                    f.write(f"    - {os.path.basename(csv_path)}\n")
            f.write("\n")
    print(f"{meta_path} is created/updated!")

write_project_info_file(
    project_dir=directory,
    project_name=project_name,
    groups=groups,
    conditions=conditions,
    filenames=filenames,
)

../processed_dataset/project_SP_ProglumideSNI/raw_data_project_SP_ProglumideSNI.pkl is created and saved!


In [5]:
###### CHECK DATA STORED CORRECTLY IN DICTIONARY ######
from utils.classification import load_behaviors, load_data
data = load_data(f"../processed_dataset/{project_name}/raw_data_{project_name}.pkl")
data

{'NoInjury': {'Male': {'M17_20241117DLC_resnet50_LUPE_MALEDec5shuffle1_350000': array([[640.13895464, 443.13371921, 639.07411766, ..., 188.78485036,
           573.77549434, 175.11807275],
          [641.59908676, 443.21913326, 642.25619805, ..., 190.06787932,
           565.33934665, 171.16458535],
          [643.10969705, 441.49057275, 643.52207786, ..., 192.52242523,
           564.23589993, 172.33931971],
          ...,
          [462.64025751, 643.84235406, 454.78470397, ..., 612.96041045,
           293.64987087, 568.51050186],
          [464.70900249, 639.47745347, 459.21974492, ..., 612.45907903,
           294.19562227, 566.61394906],
          [467.61615777, 637.74232316, 463.36172009, ..., 615.12588477,
           291.73883677, 571.81038904]]),
   'M15_20241117DLC_resnet50_LUPE_MALEDec5shuffle1_350000': array([[475.04828793, 102.0372138 , 463.38655329, ..., 107.18711257,
           218.09068966, 123.41606355],
          [474.74208748, 104.23131275, 461.52028501, ..., 106.722

# MOVE TO STEP 2!
2_preprocess_get_features.ipynb 